# Optimización de centroides M2 — cobertura marginal (anti-solapamiento)

Para cada hub M2 busca la posición dentro de `SEARCH_RADIUS` que maximiza las
cargas cubiertas de forma **exclusiva** (no ya cubiertas por otro hub).
Esto evita que hubs vecinos converjan hacia los mismos clústeres.

Los hubs se optimizan en orden de importancia (más cargas primero).
Después de mover cada hub se actualiza su posición para que el siguiente
no invada su territorio.

**Input hubs**: `final_results.csv` (192 hubs; los 4 imputados NJ/CT/RI/DE se excluyen antes de optimizar)
**Input coords**: `coords_assign_m2.parquet`
**Input cargas**: `../2. cleaning_loads/parquetloadsfull.parquet`
**Output**: `final_results_optimized.csv` (188 hubs orgánicos con centroides optimizados)

### Metadata del análisis
| Campo | Valor |
|---|---|
| N registros | 57,573,489 |
| Ventana de fechas | Feb–Ago 2025 |
| Tipo de validación | Within-sample |
| Input hubs | `final_results.csv` — 192 hubs M2 (los 4 imputados NJ/CT/RI/DE se excluyen antes de optimizar) |
| Output hubs | `final_results_optimized.csv` — **188 hubs M2 orgánicos** con centroides optimizados |

In [ ]:
import pandas as pd
import numpy as np
import polars as pl
from tqdm.auto import tqdm

LOADS_PATH      = '../2. cleaning_loads/parquetloadsfull.parquet'
HUBS_PATH       = 'final_results.csv'
COORDS_PATH     = 'coords_assign_m2.parquet'

SEARCH_RADIUS   = 80    # millas máximas que puede moverse cada hub
COVERAGE_RADIUS = 50    # radio de cobertura que se quiere maximizar
GRID_SIZE       = 15    # grid_size x grid_size = 225 candidatos + posición original

# Hubs imputados manualmente solo para el pipeline de indicadores (no para comparación de métodos)
IMPUTED_STATES  = {'NJ', 'CT', 'RI', 'DE'}


## 1. Cargar hubs M2

In [ ]:
hubs_raw = pd.read_csv(HUBS_PATH)

rows = []
for i, (_, r) in enumerate(hubs_raw.dropna(subset=['lat', 'lng']).iterrows()):
    if r.get('state') in IMPUTED_STATES:
        continue
    if pd.notna(r.get('new_name')):
        hub_id = str(r['id'])
    else:
        hub_id = f'C{i}' if pd.notna(r.get('cluster')) else None
    if hub_id is not None:
        rows.append({'hub_id': hub_id,
                     'hub_name': r.get('new_name') or hub_id,
                     'lat_orig': float(r['lat']),
                     'lng_orig': float(r['lng'])})

hubs = pd.DataFrame(rows).reset_index(drop=True)
print(f'Hubs M2 (sin imputados NJ/CT/RI/DE): {len(hubs)}')
hubs.head()


## 2. Cargar coordenadas únicas y pesos (# cargas por coordenada)

In [3]:
# Coordenadas únicas del parquet de asignación
coords_unique = pl.read_parquet(COORDS_PATH).select(['lat', 'lng'])

# Contar cargas por coordenada (origen + destino)
orig = (
    pl.scan_parquet(LOADS_PATH)
    .select([
        pl.col('LatOrigin').cast(pl.Float64).round(6).alias('lat'),
        pl.col('LngOrigin').cast(pl.Float64).round(6).alias('lng'),
    ])
    .filter(pl.col('lat').is_not_null() & pl.col('lng').is_not_null())
    .group_by(['lat', 'lng'])
    .agg(pl.len().alias('n_orig'))
    .collect()
)
dest = (
    pl.scan_parquet(LOADS_PATH)
    .select([
        pl.col('LatDestination').cast(pl.Float64).round(6).alias('lat'),
        pl.col('LngDestination').cast(pl.Float64).round(6).alias('lng'),
    ])
    .filter(pl.col('lat').is_not_null() & pl.col('lng').is_not_null())
    .group_by(['lat', 'lng'])
    .agg(pl.len().alias('n_dest'))
    .collect()
)

coords = (
    coords_unique
    .join(orig,  on=['lat', 'lng'], how='left')
    .join(dest,  on=['lat', 'lng'], how='left')
    .with_columns([
        pl.col('n_orig').fill_null(0),
        pl.col('n_dest').fill_null(0),
    ])
    .with_columns([
        (pl.col('n_orig') + pl.col('n_dest')).alias('weight')
    ])
)

coords_lat    = coords['lat'].to_numpy()
coords_lng    = coords['lng'].to_numpy()
coords_weight = coords['weight'].to_numpy().astype(float)

print(f'Coordenadas: {len(coords_lat):,}')
print(f'Cargas totales ponderadas: {coords_weight.sum():,.0f}')
print(f'Peso promedio por coord: {coords_weight.mean():.1f}')


Coordenadas: 738,827
Cargas totales ponderadas: 114,084,141
Peso promedio por coord: 154.4


## 3. Función de optimización ponderada

In [4]:
def optimize_hub(hub_lat, hub_lng, all_lats, all_lngs, weights,
                 other_lats, other_lngs,
                 search_radius=SEARCH_RADIUS,
                 coverage_radius=COVERAGE_RADIUS,
                 grid_size=GRID_SIZE):
    # Pre-filtrar coords dentro de bounding box
    margin = (search_radius + coverage_radius) / 69.0
    mask = (
        (all_lats >= hub_lat - margin) & (all_lats <= hub_lat + margin) &
        (all_lngs >= hub_lng - margin * 1.3) & (all_lngs <= hub_lng + margin * 1.3)
    )
    local_lats    = all_lats[mask]
    local_lngs    = all_lngs[mask]
    local_weights = weights[mask]
    if len(local_lats) == 0:
        return hub_lat, hub_lng, 0

    # Coords ya cubiertas por OTROS hubs (posiciones actuales)
    if len(other_lats) > 0:
        o_lat = np.radians(other_lats)[:, None]
        o_lng = np.radians(other_lngs)[:, None]
        p_lat = np.radians(local_lats)[None, :]
        p_lng = np.radians(local_lngs)[None, :]
        dlat = p_lat - o_lat
        dlng = p_lng - o_lng
        a = np.sin(dlat/2)**2 + np.cos(o_lat)*np.cos(p_lat)*np.sin(dlng/2)**2
        d_other = 3958.8 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
        already_covered = (d_other <= coverage_radius).any(axis=0)   # (n_local,)
    else:
        already_covered = np.zeros(len(local_lats), dtype=bool)

    # Solo puntuar cargas que este hub cubriría de forma exclusiva
    unique_weights = local_weights * (~already_covered)

    # Generar grilla de candidatos + posición original
    lat_step = search_radius / 69.0
    lng_step = search_radius / (69.0 * max(np.cos(np.radians(hub_lat)), 0.1))
    cand_lats = np.linspace(hub_lat - lat_step, hub_lat + lat_step, grid_size)
    cand_lngs = np.linspace(hub_lng - lng_step, hub_lng + lng_step, grid_size)
    grid_lat, grid_lng = np.meshgrid(cand_lats, cand_lngs)
    cand_lats_flat = np.append(grid_lat.flatten(), hub_lat)
    cand_lngs_flat = np.append(grid_lng.flatten(), hub_lng)

    # Distancias vectorizadas: (n_candidatos, n_coords_locales)
    c_lat = np.radians(cand_lats_flat)[:, None]
    c_lng = np.radians(cand_lngs_flat)[:, None]
    p_lat = np.radians(local_lats)[None, :]
    p_lng = np.radians(local_lngs)[None, :]
    dlat  = p_lat - c_lat
    dlng  = p_lng - c_lng
    a     = np.sin(dlat/2)**2 + np.cos(c_lat)*np.cos(p_lat)*np.sin(dlng/2)**2
    dist  = 3958.8 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    # Score = cargas únicas cubiertas (marginal coverage)
    scores = (dist <= coverage_radius) @ unique_weights
    best   = int(scores.argmax())
    return float(cand_lats_flat[best]), float(cand_lngs_flat[best]), float(scores[best])

## 4. Optimizar todos los hubs

In [5]:
# Precomputar importancia inicial de cada hub (cargas dentro de COVERAGE_RADIUS original)
# → los más importantes se optimizan primero y "reclaman" su territorio
init_scores = []
h_lats_r = np.radians(hubs['lat_orig'].values)
h_lngs_r = np.radians(hubs['lng_orig'].values)
c_lats_r  = np.radians(coords_lat)
c_lngs_r  = np.radians(coords_lng)

for idx in range(len(hubs)):
    dlat = c_lats_r - h_lats_r[idx]
    dlng = c_lngs_r - h_lngs_r[idx]
    a    = np.sin(dlat/2)**2 + np.cos(h_lats_r[idx])*np.cos(c_lats_r)*np.sin(dlng/2)**2
    d    = 3958.8 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    init_scores.append((d <= COVERAGE_RADIUS) @ coords_weight)

hubs['init_score'] = init_scores
order = hubs.sort_values('init_score', ascending=False).index.tolist()
print(f'Top 5 hubs por cargas iniciales:')
print(hubs.loc[order[:5], ['hub_name', 'lat_orig', 'lng_orig', 'init_score']].to_string())

# Posiciones actuales (se actualizan conforme optimizamos)
opt_lats = hubs['lat_orig'].values.copy()
opt_lngs = hubs['lng_orig'].values.copy()

results_map = {}
for idx in tqdm(order):
    row = hubs.loc[idx]

    # Otros hubs: sus posiciones ya optimizadas (o originales si aún no fueron)
    other_mask  = np.ones(len(hubs), dtype=bool)
    other_mask[idx] = False
    other_lats  = opt_lats[other_mask]
    other_lngs  = opt_lngs[other_mask]

    opt_lat, opt_lng, score = optimize_hub(
        row['lat_orig'], row['lng_orig'],
        coords_lat, coords_lng, coords_weight,
        other_lats, other_lngs
    )

    # Actualizar posición para que el siguiente hub la respete
    opt_lats[idx] = opt_lat
    opt_lngs[idx] = opt_lng

    # Distancia de desplazamiento
    dlat = np.radians(opt_lat - row['lat_orig']) / 2
    dlng = np.radians(opt_lng - row['lng_orig']) / 2
    a_mv = (np.sin(dlat)**2 +
            np.cos(np.radians(row['lat_orig'])) * np.cos(np.radians(opt_lat)) * np.sin(dlng)**2)
    moved_mi = round(3958.8 * 2 * np.arctan2(np.sqrt(a_mv), np.sqrt(1 - a_mv)), 1)

    results_map[idx] = {
        'hub_id':        row['hub_id'],
        'hub_name':      row['hub_name'],
        'lat_orig':      row['lat_orig'],
        'lng_orig':      row['lng_orig'],
        'lat_opt':       opt_lat,
        'lng_opt':       opt_lng,
        'loads_50m_opt': score,
        'moved_mi':      moved_mi,
    }

opt_df = pd.DataFrame([results_map[i] for i in range(len(hubs))])
print(f'Hubs que se movieron (> 0.5 mi): {(opt_df["moved_mi"] > 0.5).sum()} / {len(opt_df)}')
print(f'Desplazamiento promedio: {opt_df["moved_mi"].mean():.1f} mi')
print(f'Desplazamiento máximo:   {opt_df["moved_mi"].max():.1f} mi')
opt_df.sort_values('moved_mi', ascending=False).head(10)

Top 5 hubs por cargas iniciales:
              hub_name   lat_orig   lng_orig  init_score
21             Pico IL  41.893900 -87.951229   3901949.0
72             Pico TX  32.853836 -96.990061   3841773.0
46  New Brunswick (NJ)  40.486200 -74.451800   3275618.0
15             Pico GA  33.697806 -84.482410   3031849.0
73             Pico TX  29.850597 -95.407468   2921274.0


100%|████████████████████████████████████████████████████████████████████████████████| 192/192 [00:51<00:00,  3.71it/s]

Hubs que se movieron (> 0.5 mi): 189 / 192
Desplazamiento promedio: 37.2 mi
Desplazamiento máximo:   113.9 mi


,hub_id,hub_name,lat_orig,lng_orig,lat_opt,lng_opt,loads_50m_opt,moved_mi
137,C137,NaN,45.205340,-112.644640,44.045920,-110.999064,8706.0,113.9
124,C124,NaN,44.720199,-85.650100,43.560779,-84.018381,190712.0,113.9
87,78.0,Pico WY,41.569278,-109.350351,40.409857,-110.900059,59553.0,113.8
120,C120,NaN,37.741916,-99.967494,36.582496,-98.501313,61578.0,113.7
168,C168,NaN,31.714701,-99.966300,30.555280,-98.603360,80258.0,113.6
158,C158,NaN,35.445599,-98.977600,34.286178,-97.757728,138923.0,105.8
101,C101,NaN,37.999100,-103.592000,37.170943,-105.063307,22164.0,98.8
121,C121,NaN,38.405141,-96.225392,37.411352,-94.957218,84997.0,97.4
95,C95,NaN,39.389739,-123.336280,40.383528,-122.050400,138970.0,96.8
132,C132,NaN,31.552999,-91.371398,32.546788,-92.343235,336180.0,89.2


## 5. Guardar `final_results_optimized.csv`

In [ ]:
out = hubs_raw[~hubs_raw['state'].isin(IMPUTED_STATES)].dropna(subset=['lat', 'lng']).copy().reset_index(drop=True)
out['lat'] = opt_df['lat_opt'].values
out['lng'] = opt_df['lng_opt'].values

out.to_csv('final_results_optimized.csv', index=False)
print(f'Guardado: final_results_optimized.csv  ({len(out)} hubs, sin imputados NJ/CT/RI/DE)')
out.head()

## 6. Mapa: posiciones originales vs optimizadas

In [8]:
import folium
from IPython.display import display

m = folium.Map(location=[38, -96], zoom_start=4, tiles='CartoDB positron')

for _, row in opt_df.iterrows():
    if row['moved_mi'] > 0.5:
        # Flecha: línea de original a optimizado
        folium.PolyLine(
            [[row['lat_orig'], row['lng_orig']], [row['lat_opt'], row['lng_opt']]],
            color='orange', weight=1.5, opacity=0.7
        ).add_to(m)
    # Posición original (gris)
    folium.CircleMarker(
        [row['lat_orig'], row['lng_orig']],
        radius=3, color='gray', fill=True, fill_opacity=0.4,
        tooltip=f"{row['hub_name']} (original)"
    ).add_to(m)
    # Posición optimizada (azul)
    folium.CircleMarker(
        [row['lat_opt'], row['lng_opt']],
        radius=4, color='steelblue', fill=True, fill_opacity=0.8,
        tooltip=f"{row['hub_name']} → {row['moved_mi']:.1f} mi"
    ).add_to(m)

display(m)


/home/karen/hubs/jupyter-env/lib/python3.11/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
